In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const
from astropy.table import Table
from astropy.cosmology import Planck18
import requests
from io import StringIO
from astropy.io import fits
from astropy.visualization import simple_norm
import os
from glob import glob
from SciServer import CasJobs

df = pd.read_csv('../all_sample_20260702.csv')
df

,paliya_RA,paliya_DEC,paliya_Z
0,0.436896,11.166670,0.087600
1,1.417268,0.318698,0.084819
2,4.243540,7.345316,0.085980
3,4.264051,-4.789808,0.098000
4,4.718625,1.132922,0.063872
...,...,...,...
487,348.070200,-10.236200,0.065636
488,351.679500,-9.991077,0.069534
489,356.120100,13.829460,0.068452
490,357.720900,33.130630,0.091844


In [2]:
z_value = df['paliya_Z'].values
df = df.rename(columns = {"paliya_RA":"RA", "paliya_DEC":"DEC"})

distance = Planck18.comoving_distance(z_value).value
size_ang = np.rad2deg(np.arcsin(0.5/2/distance))
size = np.astype(np.rad2deg(np.arcsin(0.5/2/distance)*2)*3600*4, 'int')

## Get Img Data

In [ ]:
"""ps1bulk.py: Get PS1 stack FITS cutout images at a list of positions

NOTE: If you modify this script to download images in multiple threads,
please do not use more than 10 simultaneous threads for the download.
The ps1images service is a shared resource, and too many requests from
a single user can cause the system to be unresponsive for all users.
If you attempt to download images at an excessive rate, eventually you
will find your downloads blocked by the server.
"""

ps1filename = "https://ps1images.stsci.edu/cgi-bin/ps1filenames.py"
fitscut = "https://ps1images.stsci.edu/cgi-bin/fitscut.cgi"

def getimages(tra, tdec, size, filters="grizy", format="fits", imagetypes="stack"):

    """Query ps1filenames.py service for multiple positions to get a list of images
    This adds a url column to the table to retrieve the cutout.

    tra, tdec = list of positions in degrees
    size = image size in pixels (0.25 arcsec/pixel)
    filters = string with filters to include
    format = data format (options are "fits", "jpg", or "png")
    imagetypes = list of any of the acceptable image types.  Default is stack;
        other common choices include warp (single-epoch images), stack.wt (weight image),
        stack.mask, stack.exp (exposure time), stack.num (number of exposures),
        warp.wt, and warp.mask.  This parameter can be a list of strings or a
        comma-separated string.

    Note that it is much faster to provide a list of positions if you have many positions
    to query! For example, querying 1000 positions one at a time takes approximately 8
    minutes, while uploading 1000 positions in a single query takes less than 1.5 seconds.
    If you are doing large-scale queries, you absolutely should be using the ability to
    upload position lists.

    Returns an astropy table with the results
    """

    if format not in ("jpg","png","fits"):
        raise ValueError("format must be one of jpg, png, fits")
    # if imagetypes is a list, convert to a comma-separated string
    if not isinstance(imagetypes,str):
        imagetypes = ",".join(imagetypes)
    # put the positions in an in-memory file object
    cbuf = StringIO()
    cbuf.write('\n'.join(["{} {}".format(ra, dec) for (ra, dec) in zip(tra,tdec)]))
    cbuf.seek(0)
    # use requests.post to pass in positions as a file
    r = requests.post(ps1filename, data=dict(filters=filters, type=imagetypes),
        files=dict(file=cbuf))
    r.raise_for_status()
    tab = Table.read(r.text, format="ascii")

    urlbase = ["{}?size={}&format={}".format(fitscut,s,format) for (s) in size]
    urlbase = [tmp for tmp in urlbase for i in range(len(filters))]
    tab["url"] = ["{}&ra={}&dec={}&red={}".format(url,ra,dec,filename)
            for (url,filename,ra,dec) in zip(urlbase, tab["filename"],tab["ra"],tab["dec"])]
    return tab

def release():
    import gc
    # 1. 透過 globals() 字典安全地刪除全域變數
    if 'data' in globals():
        del globals()['data']
    if 'header' in globals():
        del globals()['header']
    # 2. 強制進行垃圾回收
    gc.collect()

if __name__ == "__main__":
    release()
    # get the PS1 info for those positions
    table = getimages(df['paliya_RA'][:5],df['paliya_DEC'][:5], size[:5])
    # if you are extracting images that are close together on the sky,
    # sorting by skycell and filter will improve the performance because it takes
    # advantage of file system caching on the server
    table.sort(['projcell','subcell','filter'])

    # extract cutout for each position/filter combination
    for row in table:
        ra = row['ra']
        dec = row['dec']
        projcell = row['projcell']
        subcell = row['subcell']
        filter = row['filter']

        # create a name for the image -- could also include the projection cell or other info
        fname = "../data/img/t{:08.4f}{:+07.4f}/t{:08.4f}{:+07.4f}.{}.fits".format(ra,dec,ra,dec,filter)

        url = row["url"]
        print("%11.6f %10.6f skycell.%4.4d.%3.3d %s" % (ra, dec, projcell, subcell, fname))
        r = requests.get(url)
        try:
            open(fname,"wb").write(r.content)
        except:
            try:
                print('Creating Path...')
                os.makedirs("../data/img/t{:08.4f}{:+07.4f}".format(ra,dec), exist_ok = True)
                open(fname,"wb").write(r.content)
            except:
                print('Creating Path Failed')

## Get Catalog

In [ ]:
import requests
from io import StringIO
from astropy.table import Table

# 1. Change the base URL from the image server to the catalog server
PS1_CATALOG_URL = "https://catalogs.mast.stsci.edu/api/v0.1/panstarrs/dr2/mean"

def getcatalog(ra, dec, radius_degrees):
    """Query the live PS1 catalog for all objects within a specific region.
    
    ra, dec = target center position in decimal degrees
    radius_degrees = the constant region size in degrees
    """
    # The Catalog API requires the radius parameter to be in decimal degrees 
    # Setup the query parameters for the server request
    query_params = {
        "ra": ra,
        "dec": dec,
        "radius": radius_degrees,
        "format": "csv"
    }
    
    # Ping the server using requests.get, exactly like your original script
    r = requests.get(PS1_CATALOG_URL, params=query_params)
    r.raise_for_status()
    
    # Read into an astropy Table, just like Table.read(r.text, format="ascii")
    tab = Table.read(r.text, format="csv")
    return tab

if __name__ == "__main__":
    # Loop through your exact target arrays from your dataframe
    for ra, dec, ang in zip(df['RA'], df['DEC'], size_ang):
        
        # Get the catalog table for this specific region view
        table = getcatalog(ra, dec, radius_degrees= ang)
        # table.keep_columns(['objID', 'raMean', 'decMean'])
        # table.rename_columns(['raMean', 'decMean'], ['ra', 'dec'])

        # Define a name for the spreadsheet file matching the coordinates
        fname = "../data/catalog/ps1/t{:08.4f}{:+07.4f}.csv".format(ra, dec)
        
        # Save the full target region catalog locally to disk
        table.write(fname, format="csv", overwrite=True)
        print("Successfully saved catalog containing {} rows to {}".format(len(table), fname))

Successfully saved catalog containing 1168 rows to ../data/catalog/ps1/t000.4369+11.1667.csv
Successfully saved catalog containing 2058 rows to ../data/catalog/ps1/t001.4173+0.3187.csv
Successfully saved catalog containing 1513 rows to ../data/catalog/ps1/t004.2435+7.3453.csv
Successfully saved catalog containing 1285 rows to ../data/catalog/ps1/t004.2641-4.7898.csv
Successfully saved catalog containing 3093 rows to ../data/catalog/ps1/t004.7186+1.1329.csv
Successfully saved catalog containing 1328 rows to ../data/catalog/ps1/t005.0301+32.7363.csv
Successfully saved catalog containing 3481 rows to ../data/catalog/ps1/t005.6872-3.0585.csv
Successfully saved catalog containing 10456 rows to ../data/catalog/ps1/t006.0796-1.6382.csv
Successfully saved catalog containing 2288 rows to ../data/catalog/ps1/t006.1904+8.3492.csv
Successfully saved catalog containing 1512 rows to ../data/catalog/ps1/t007.0451+15.9338.csv
Successfully saved catalog containing 2491 rows to ../data/catalog/ps1/t008.

In [ ]:
import pandas as pd
import requests
from io import StringIO
from astropy.table import Table
from concurrent.futures import ThreadPoolExecutor, as_completed

PS1_CATALOG_URL = "https://catalogs.mast.stsci.edu/api/v0.1/panstarrs/dr2/mean"

def getcatalog(ra, dec, radius_degrees):
    """維持您原本的 GET 查詢邏輯"""
    query_params = {
        "ra": ra,
        "dec": dec,
        "radius": radius_degrees,
        "format": "csv"
    }
    # 這裡完全沿用您原本的 requests.get
    r = requests.get(PS1_CATALOG_URL, params=query_params)
    r.raise_for_status()
    
    return Table.read(r.text, format="csv")

if __name__ == "__main__":
    df_training_sample = pd.read_csv('../data/training/training_data.csv')
    
    # 準備您的陣列片段
    ra_list = df_training_sample['ra'].tolist()
    dec_list = df_training_sample['dec'].tolist()
    radius = 1/3600
    
    # 使用 Thread 批次並行訪問（同時發出 5 個 GET 請求）
    with ThreadPoolExecutor(max_workers=20) as executor:
        # 將任務分派出去，這行執行完時，5 個 GET 請求已經同時出發了
        futures = {
            executor.submit(getcatalog, ra, dec, radius): (ra, dec) 
            for ra, dec in zip(ra_list, dec_list)
        }
        
        # 哪一個請求先回來，就先處理哪一個
        for future in as_completed(futures):
            orig_ra, orig_dec = futures[future]
            try:
                table = future.result()
                
                if len(table) == 0:
                    print(f"沒有找到對應星體: RA={orig_ra}, DEC={orig_dec}")
                    continue
                    
                # 執行您的欄位清理與篩選
                table.keep_columns(['objID', 'raMean', 'decMean'])
                table.rename_columns(['raMean', 'decMean'], ['ra', 'dec'])
                
                # 依您原來的格式儲存成獨立檔案
                fname = f"../data/catalog/train_ps1/t{orig_ra:08.4f}{orig_dec:+07.4f}.csv"
                table.write(fname, format="csv", overwrite=True)
                print(f"成功並行取得並儲存: {fname}")
                
            except Exception as e:
                print(f"處理失敗 RA: {orig_ra}, DEC: {orig_dec}。錯誤原因: {e}")


成功並行取得並儲存: ../data/catalog/train_ps1/t358.1688-11.1415.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t359.1830-11.1784.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.2510-11.1562.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t359.0850-11.1166.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t359.0845-11.1344.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.9728-11.1196.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t359.9974-11.1921.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t357.9074-11.1742.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t000.5171-11.2273.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.5770-11.1303.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.6636-11.1483.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t001.3855-11.1522.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t359.9498-11.1825.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.0772-11.1710.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t358.7694-11.1259.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t000.2001-11.2007.csv
成功並行取得並儲存: ../data/catalog/train_ps1/t002.4201-11.1770.c

In [14]:
table.columns

<TableColumns names=('objName','objAltName1','objAltName2','objAltName3','objID','uniquePspsOBid','ippObjID','surveyID','htmID','zoneID','tessID','projectionID','skyCellID','randomID','batchID','dvoRegionID','processingVersion','objInfoFlag','astrometryCorrectionFlag','qualityFlag','raStack','decStack','raStackErr','decStackErr','raMean','decMean','raMeanErr','decMeanErr','pmra','pmdec','pmraErr','pmdecErr','epochMean','posMeanChisq','cx','cy','cz','lambda','beta','l','b','nStackObjectRows','nStackDetections','nDetections','ng','nr','ni','nz','ny','gQfPerfect','gMeanPSFMag','gMeanPSFMagErr','gMeanPSFMagStd','gMeanPSFMagNpt','gMeanPSFMagMin','gMeanPSFMagMax','gMeanKronMag','gMeanKronMagErr','gMeanKronMagStd','gMeanKronMagNpt','gMeanApMag','gMeanApMagErr','gMeanApMagStd','gMeanApMagNpt','gFlags','rQfPerfect','rMeanPSFMag','rMeanPSFMagErr','rMeanPSFMagStd','rMeanPSFMagNpt','rMeanPSFMagMin','rMeanPSFMagMax','rMeanKronMag','rMeanKronMagErr','rMeanKronMagStd','rMeanKronMagNpt','rMeanApMag','

## Get Redshift

In [6]:
def get_data_SDSS(target_ra, target_dec, radius_arcmin, directory = 'redshift'):
    import pandas as pd
    from SciServer import CasJobs

    # Define your target
    radius_arcmin = radius_arcmin*60   

    # 5. Inject the unrestricted column string into the radial search query
    photo_sql_query = f"""
    SELECT 
        n.distance, 
        p.objid, p.ra, p.dec, p.petroMag_g, p.petroMag_r, p.petroMag_i, p.petroMag_z,
        pz.z AS photo_redshift,      -- The estimated redshift of the environment galaxy
        pz.zErr AS redshift_error    -- The error you need for your Gaussian weights!
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid  -- Pulls from the machine-learning redshift table
    WHERE 
        p.type = 3        -- 3 is the SDSS code for 'GALAXY' (replaces s.class = 'GALAXY')
        AND p.clean = 1   -- Ensures no bad pixels
    ORDER BY n.distance ASC
    """

    spectrum_sql_query = f"""
    SELECT 
        n.distance,
        s.specObjID, 
        s.z AS redshift
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN SpecObj AS s ON s.bestobjid = n.objID
    JOIN galSpecLine AS g ON s.specObjID = g.specObjID
    WHERE 
        s.zWarning = 0  
    ORDER BY n.distance ASC
    """

    both_sql_query = f"""
    SELECT
        n.distance, 
        p.objid, p.ra, p.dec, p.petroMag_u, p.petroMag_g, p.petroMag_r, p.petroMag_i, p.petroMag_z,
        p.petroMagErr_u, p.petroMagErr_g, p.petroMagErr_r, p.petroMagErr_i, p.petroMagErr_z,
        pz.z AS photo_redshift,
        pz.zErr AS photo_redshift_error, 
        s.specObjID, s.z AS redshift
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid
    JOIN SpecObj AS s ON s.bestobjid = n.objID
    JOIN galSpecLine AS g ON s.specObjID = g.specObjID
    WHERE
        p.type = 3
        AND p.clean = 1
        AND s.zWarning = 0
    ORDER BY n.distance ASC
    """
    try_sql_query = f"""
    SELECT 
        n.distance, 
        p.objid, p.ra, p.dec, 
        
        -- 1. Get cModel Magnitudes for accurate flux/colors
        p.cModelMag_u, p.cModelMag_g, p.cModelMag_r, p.cModelMag_i, p.cModelMag_z,
        p.cModelMagErr_u, p.cModelMagErr_g, p.cModelMagErr_r, p.cModelMagErr_i, p.cModelMagErr_z,
        
        -- 2. Get Galactic Extinction for dereddening
        p.extinction_u, p.extinction_g, p.extinction_r, p.extinction_i, p.extinction_z,
        
        -- 3. SDSS pre-calculated Photo-Z and Error Classes
        pz.z AS sdss_photo_redshift,
        pz.zErr AS sdss_photo_redshift_error,
        pz.photoErrorClass,
        
        -- 4. True Spectroscopic Redshift (Ground Truth)
        s.specObjID, 
        s.z AS spec_redshift, 
        s.zErr AS spec_redshift_error

    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid
    JOIN SpecObj AS s ON s.bestobjid = p.objid

    WHERE
        p.type = 3                -- Ensure Photometric object is a Galaxy
        AND p.clean = 1           -- Ensure Photometry is not corrupted by dead pixels/edges
        AND s.class = 'GALAXY'    -- Ensure Spectroscopic object is a Galaxy
        AND s.zWarning = 0        -- Ensure Spectroscopic Redshift is 100% reliable
        
    ORDER BY n.distance ASC
    """

    get_data_sql_query = f"""
    SELECT 
        n.distance, 
        p.objid, p.ra, p.dec, 
        
        -- 1. Get cModel Magnitudes for accurate flux/colors
        p.cModelMag_u, p.cModelMag_g, p.cModelMag_r, p.cModelMag_i, p.cModelMag_z,
        p.cModelMagErr_u, p.cModelMagErr_g, p.cModelMagErr_r, p.cModelMagErr_i, p.cModelMagErr_z,
        
        -- 2. Get Galactic Extinction for dereddening
        p.extinction_u, p.extinction_g, p.extinction_r, p.extinction_i, p.extinction_z

    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid
    JOIN SpecObj AS s ON s.bestobjid = p.objid

    WHERE
        p.type = 3                -- Ensure Photometric object is a Galaxy
        AND p.clean = 1           -- Ensure Photometry is not corrupted by dead pixels/edges
        AND s.class = 'GALAXY'    -- Ensure Spectroscopic object is a Galaxy
        AND s.zWarning = 0        -- Ensure Spectroscopic Redshift is 100% reliable
        
    ORDER BY n.distance ASC
    """


    try:
        # 6. Execute the massive query via SciServer CasJobs
        df_sdss = CasJobs.executeQuery(sql=get_data_sql_query, context="DR19", format="pandas")
        
        # Save it 
        df_sdss.to_csv(f"../data/{directory}/{target_ra:4f}-{target_dec:4f}-{radius_arcmin:4f}.csv", index=False)
        
        print("\n--- SUCCESS ---")
        print(f"Downloaded exactly {len(df_sdss)} columns!")
        # print(df_sdss.head())

    except Exception as e:
        print(f"An error occurred: {e}")

In [76]:
def get_data_SDSS_longer(target_ra, target_dec, radius_arcmin, directory = 'redshift'):
    import pandas as pd
    from SciServer import CasJobs

    # Define your target
    radius_arcmin = radius_arcmin*60   

    # 5. Inject the unrestricted column string into the radial search query
    photo_sql_query = f"""
    SELECT 
        n.distance, 
        p.objid, p.ra, p.dec, p.petroMag_g, p.petroMag_r, p.petroMag_i, p.petroMag_z,
        pz.z AS photo_redshift,      -- The estimated redshift of the environment galaxy
        pz.zErr AS redshift_error    -- The error you need for your Gaussian weights!
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid  -- Pulls from the machine-learning redshift table
    WHERE 
        p.type = 3        -- 3 is the SDSS code for 'GALAXY' (replaces s.class = 'GALAXY')
        AND p.clean = 1   -- Ensures no bad pixels
    ORDER BY n.distance ASC
    """

    spectrum_sql_query = f"""
    SELECT 
        n.distance,
        s.specObjID, 
        s.z AS redshift
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN SpecObj AS s ON s.bestobjid = n.objID
    JOIN galSpecLine AS g ON s.specObjID = g.specObjID
    WHERE 
        s.zWarning = 0  
    ORDER BY n.distance ASC
    """

    both_sql_query = f"""
    SELECT
        n.distance, 
        p.objid, p.ra, p.dec, p.petroMag_u, p.petroMag_g, p.petroMag_r, p.petroMag_i, p.petroMag_z,
        p.petroMagErr_u, p.petroMagErr_g, p.petroMagErr_r, p.petroMagErr_i, p.petroMagErr_z,
        pz.z AS photo_redshift,
        pz.zErr AS photo_redshift_error, 
        s.specObjID, s.z AS redshift
    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid
    JOIN SpecObj AS s ON s.bestobjid = n.objID
    JOIN galSpecLine AS g ON s.specObjID = g.specObjID
    WHERE
        p.type = 3
        AND p.clean = 1
        AND s.zWarning = 0
    ORDER BY n.distance ASC
    """
    try_sql_query = f"""
    SELECT
        n.distance, 
        p.objid, p.ra, p.dec, 
        
        -- 1. Get cModel Magnitudes for accurate flux/colors
        p.cModelMag_u, p.cModelMag_g, p.cModelMag_r, p.cModelMag_i, p.cModelMag_z,
        p.cModelMagErr_u, p.cModelMagErr_g, p.cModelMagErr_r, p.cModelMagErr_i, p.cModelMagErr_z,
        
        -- 2. Get Galactic Extinction for dereddening
        p.extinction_u, p.extinction_g, p.extinction_r, p.extinction_i, p.extinction_z,
        
        -- 3. SDSS pre-calculated Photo-Z and Error Classes
        pz.z AS sdss_photo_redshift,
        pz.zErr AS sdss_photo_redshift_error,
        pz.photoErrorClass,
        
        -- 4. True Spectroscopic Redshift (Ground Truth)
        s.specObjID, 
        s.z AS spec_redshift, 
        s.zErr AS spec_redshift_error

    INTO MYDB.my_output_table  -- Save results to your personal database

    FROM dbo.fGetNearbyObjEq({target_ra}, {target_dec}, {radius_arcmin}) AS n
    JOIN PhotoObj AS p ON p.objid = n.objID
    JOIN Photoz AS pz ON pz.objid = p.objid
    JOIN SpecObj AS s ON s.bestobjid = p.objid

    WHERE
        p.type = 3                -- Ensure Photometric object is a Galaxy
        AND p.clean = 1           -- Ensure Photometry is not corrupted by dead pixels/edges
        AND s.class = 'GALAXY'    -- Ensure Spectroscopic object is a Galaxy
        AND s.zWarning = 0        -- Ensure Spectroscopic Redshift is 100% reliable
        
    ORDER BY n.distance ASC
    """


    try:
        # 6. Execute the massive query via SciServer CasJobs
        df_sdss = CasJobs.submitJob(sql=try_sql_query, context="DR19")
        
        # Save it 
        # df_sdss.to_csv(f"../data/{directory}/{target_ra:4f}-{target_dec:4f}-{radius_arcmin:4f}.csv", index=False)
        
        print("\n--- SUCCESS ---")
        # print(f"Downloaded exactly {len(df_sdss)} columns!")
        # print(df_sdss.head())
        return df_sdss
    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
from SciServer import Authentication
SDSS_USERNAME = "yee8787"  # e.g., "yee8787"
SDSS_PASSWORD = "4wSu;vM!D]4Dy;:"

print("Authenticating with SDSS SciServer...")
try:
    # This generates a secure token and applies it to your local environment
    token = Authentication.login(SDSS_USERNAME, SDSS_PASSWORD)
    print("Authentication successful! You now have full server privileges.")
except Exception as e:
    print(f"Login failed: {e}")
    exit()

job_id = get_data_SDSS_longer(0, -45, 1*60, 'training')

status = CasJobs.getJobStatus(job_id)
import time
time.sleep(30)
while status['Status'] == 1 and status['Status'] != 0:
    print(f"目前狀態: {status}... 10秒後重新檢查...")
    time.sleep(10)
    status = CasJobs.getJobStatus(job_id)
if status['Status'] == 5:
    print("計算完成！資料已成功寫入你的 MyDB.my_output_table")
    print("start downloading the results...")
    df = CasJobs.executeQuery(sql="SELECT * FROM MyDB.my_output_table", context="MyDB")
    df.to_csv(f"../data/training/training_data.csv", index=False)
else:
    print(f"任務未成功，最終狀態為: {status}")

Authenticating with SDSS SciServer...
Authentication successful! You now have full server privileges.

--- SUCCESS ---


In [7]:
from SciServer import Authentication
SDSS_USERNAME = "yee8787"  # e.g., "yee8787"
SDSS_PASSWORD = "4wSu;vM!D]4Dy;:"

print("Authenticating with SDSS SciServer...")
try:
    # This generates a secure token and applies it to your local environment
    token = Authentication.login(SDSS_USERNAME, SDSS_PASSWORD)
    print("Authentication successful! You now have full server privileges.")
except Exception as e:
    print(f"Login failed: {e}")
    exit()

for i, (ra, dec, ang) in enumerate(zip(df['RA'], df['DEC'], size_ang)):
    get_data_SDSS(ra, dec, ang, 'catalog')
    print(i)

Authenticating with SDSS SciServer...
Authentication successful! You now have full server privileges.

--- SUCCESS ---
Downloaded exactly 2 columns!
0

--- SUCCESS ---
Downloaded exactly 4 columns!
1

--- SUCCESS ---
Downloaded exactly 0 columns!
2

--- SUCCESS ---
Downloaded exactly 1 columns!
3

--- SUCCESS ---
Downloaded exactly 1 columns!
4

--- SUCCESS ---
Downloaded exactly 1 columns!
5

--- SUCCESS ---
Downloaded exactly 4 columns!
6

--- SUCCESS ---
Downloaded exactly 8 columns!
7

--- SUCCESS ---
Downloaded exactly 7 columns!
8

--- SUCCESS ---
Downloaded exactly 3 columns!
9

--- SUCCESS ---
Downloaded exactly 1 columns!
10

--- SUCCESS ---
Downloaded exactly 7 columns!
11

--- SUCCESS ---
Downloaded exactly 0 columns!
12

--- SUCCESS ---
Downloaded exactly 1 columns!
13

--- SUCCESS ---
Downloaded exactly 0 columns!
14

--- SUCCESS ---
Downloaded exactly 1 columns!
15

--- SUCCESS ---
Downloaded exactly 3 columns!
16

--- SUCCESS ---
Downloaded exactly 7 columns!
17

--- SUC

## img output
``` python
dec = df['paliya_DEC'].values
ra = df['paliya_RA'].values
index = 5
filt = list('grizy')

for d, r in zip(dec[:index], ra[:index]):
  fig = plt.figure(figsize = (10, 15))
  for f, _ in enumerate(filt):
    with fits.open(f'../data/img/t{r:08.4f}{d:+07.4f}/t{r:08.4f}{d:+07.4f}.{filt[f]}.fits') as hdul:
      header = hdul[0].header
      data = hdul[0].data

    from astropy.wcs import WCS
    from astropy.stats import sigma_clipped_stats, sigma_clip
    wcs = WCS(header)
    results = sigma_clipped_stats(data, sigma = 3, maxiters = 5)
    # clipped_img = sigma_clip(data)
    bkg_subtracted = data-results[1]
    denoised_data = np.where(bkg_subtracted < 3 *results[2], 0, bkg_subtracted)
    
    norm = simple_norm(denoised_data, percent = 99, stretch = 'linear')
    ax = fig.add_subplot(3, 2, f+1, projection = wcs)
    im = ax.imshow(denoised_data, cmap = 'gray', norm = norm, origin = 'lower')
    fig.colorbar(im, ax= ax)
    ax.set_title(f"Filter: {filt[f]}")
    ax.set_ylabel("DEC")
    ax.set_xlabel("RA")
  fig.suptitle(f"{r}, {d}")
  plt.tight_layout()
  plt.show()
```